# Testing Spatiotemporal Script

In [ ]:
import numpy as np
import pandas as pd
import tensorflow as tf
import matplotlib.pyplot as plt
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.model_selection import train_test_split
from scipy.io import loadmat
from scipy.io import savemat
from scipy import ndimage
import random

import mat73
import pydot
import graphviz

tf.__version__

In [ ]:
# Load data of new patients 

test_list = ['3977'] #modify
test_rest = 'rest1'  #modify

n_patient = 0
for patient_id in test_list:
    
    print('Patient:', patient_id)

    # Load data
    fmat = 'F:/AI_project/3.Preprocessing_AI/meg_'+patient_id+'/'+test_rest+'/x_coor_meg_'+patient_id+'_1x60.mat'
    x_mat = loadmat(fmat)
    fmat = 'F:/AI_project/3.Preprocessing_AI/meg_'+patient_id+'/'+test_rest+'/y_coor_meg_'+patient_id+'_1x60.mat'
    y_mat = loadmat(fmat)
    fmat = 'F:/AI_project/3.Preprocessing_AI/meg_'+patient_id+'/'+test_rest+'/z_data_meg_'+patient_id+'_xy_image_per_window.mat'
    zdata_mat = loadmat(fmat)
    fmat = 'F:/AI_project/3.Preprocessing_AI/meg_'+patient_id+'/'+test_rest+'/t_data_meg_'+patient_id+'_102xnwindows.mat'
    t_mat = loadmat(fmat)
    fmat = 'F:/AI_project/3.Preprocessing_AI/meg_'+patient_id+'/'+test_rest+'/n_windows_meg_'+patient_id+'.mat'
    windows_mat = loadmat(fmat)

    # Save data into numpy arrays
    x_coor = x_mat['vecX']
    y_coor = y_mat['vecY']
    z_data = zdata_mat['vecZ']
    t_data = t_mat['temporal_matrix_nwindow'].T  
    nw_dict = int(windows_mat['n_windows'])

    z_data = z_data[0:nw_dict]
    t_data = t_data[0:nw_dict]
    # Normalize x-data of each patient individually
    z_data = (z_data - np.nanmean(z_data)) / np.nanstd(z_data)    
    z_data = np.nan_to_num(z_data)
    t_data = (t_data - np.mean(t_data)) / np.std(t_data)
    
    print('z_data: ', z_data.shape)
    print('t_data: ', t_data.shape)

    n_patient = n_patient + 1
    print()

In [ ]:
# Load model to test it
model = tf.keras.models.load_model('F:/AI_project/4.ML_program/Model/ST_10_patients_100_trainning')
# Predictions
y_pred = model.predict([t_data,z_data])
threshold = 0.5
index_pred = np.where( y_pred[:]>threshold )[0]    

In [ ]:
positive_list = []
negative_list = []

for i in range(len(y_pred)):
    if( y_pred[i]>threshold): positive_list.append(i)
    if( y_pred[i]<threshold): negative_list.append(i)
    
print('predicted positives:', positive_list)
print(len(positive_list))
print('predicted negatives:', negative_list)
print(len(negative_list))

In [ ]:
# Original time in the rawtime data in seconds
n_timesteps = 200
spike_time_interval = []
spike_time = []
spike_time_ini = []
spike_time_fin = []

for i in range(len(positive_list)):
    spike_time_ini = n_timesteps*positive_list[i]
    spike_time_fin = spike_time_ini + n_timesteps
    spike_time_ini = spike_time_ini/1000
    spike_time_fin = spike_time_fin/1000
    spike_time_interval =  [spike_time_ini,spike_time_fin]
    spike_time.append(spike_time_interval)

print('spike timing:', spike_time)

fmat = 'F:/AI_project/4.ML_program/Results/'+test_rest+'/meg_'+patient_id+'_'+test_rest+'_ST_predicted_spikes_timing.mat'
savemat(fmat, {"spike_time": spike_time})

In [ ]:
# We select of the highest value from the temporal signal and we save the temporal real value 
amplitude_spike_max_time=[]
spike_max_time=[]
spike_real_time=[]
spikereal = 0

for i in positive_list:
    amplitude_spike_max_time = max(abs(t_data[i,:]))
    for j in range(n_timesteps):
        if( abs(t_data[i,j]) == amplitude_spike_max_time): spike_max_time.append(j)

print('index inside of each 200ms spiky window', spike_max_time)

for i in range(len(positive_list)):
    spikereal = spike_max_time[i]+n_timesteps*positive_list[i]
    spike_real_time.append(spikereal)

print('real time in ms', spike_real_time) # also can be interpreted as the index in the rawtime file

In [ ]:
# The number of spikes is:
print('Number of predicted spikes:', len(positive_list))

spike_wave_index=(len(positive_list)*200)/((len(positive_list)+len(negative_list))*200)*100
print('Spike-wave index:', spike_wave_index)

In [ ]:
# Spikes predicted 

n_timesteps = 200
i=-1
for id in index_pred:
    print(id)
    i = i + 1
    print(spike_real_time[i])
    
    plt.plot(range(n_timesteps),t_data[id,0:n_timesteps])    
    plt.show()

    plt.imshow(z_data[id,:,:], interpolation='bilinear')#,vmin=-4.0,vmax=4.0)    
    plt.colorbar()
    plt.title("Spike predicted")
    plt.show()